# Lesson 3 — PTC Extraction: From Text to Structured Claims

**Learning objective:** Understand how the PTCExtractor works — prompt versioning, the validation pipeline, deduplication, and resume-safe JSONL persistence. Run a full extraction against a mock backend (no AWS credentials required).

---

## Architecture

```
Document list
    └─▶ PTCExtractor.extract_from_documents()
            ├─▶ split_document()          (Lesson 2 chunker)
            ├─▶ LLMBackend.complete()     (Bedrock Claude in production)
            ├─▶ validate_ptc_dict()       (pydantic validation + drop-on-error)
            └─▶ append_to_pool()          (JSONL persistence, resume-safe)
```

The backend is swappable. In production it calls Bedrock. In tests and this notebook it's a mock that returns pre-crafted JSON — so you can run this lesson without AWS credentials.

In [ ]:
import sys, json, tempfile
from pathlib import Path
from datetime import date
from unittest.mock import MagicMock

sys.path.insert(0, '../../src')

from ptc_extraction.extractor import PTCExtractor
from ptc_extraction.backends.base import LLMBackend
from ptc_extraction.io import load_pool

## Exercise 1 — Build a mock backend

The `LLMBackend` abstract class requires one method: `complete(system, user) -> str`. We'll wire it to return a pre-crafted PTC array.

In [ ]:
MOCK_RESPONSE = json.dumps([
    {
        "mechanism": "Red Sea disruptions elevated logistics costs by approximately €45M and are expected to persist into mid-2025.",
        "raw_text": "Red Sea disruptions added €45M in costs over nine months; resolution not expected before mid-2025.",
        "direction": "precursor",
        "polarity": -1,
        "event_date": "2024-01-01",
        "confidence": 0.92,
    },
    {
        "mechanism": "Euro appreciation realised a €120M negative translation impact on Q3 revenue.",
        "raw_text": "The stronger euro compressed Q3 revenue by approximately €120M.",
        "direction": "consequence",
        "polarity": -1,
        "event_date": "2024-07-01",
        "confidence": 0.88,
    },
    {
        "mechanism": "Euro forex headwinds are expected to persist into Q4, continuing to compress revenue translation.",
        "raw_text": "Forex headwinds are expected to persist into Q4 given current exchange rate levels.",
        "direction": "precursor",
        "polarity": -1,
        "event_date": None,
        "confidence": 0.85,
    },
])

mock_backend = MagicMock(spec=LLMBackend)
mock_backend.model_id = "mock-model"
mock_backend.complete.return_value = MOCK_RESPONSE

print("Mock backend ready. complete() will return", len(json.loads(MOCK_RESPONSE)), "PTCs per call.")

## Exercise 2 — Run the extractor end-to-end

In [ ]:
# A single document (as the pipeline expects)
documents = [
    {
        "source_id": "ML.PA-management-2024-11-14-q3-results",
        "text": (
            "Michelin Group reported strong Q3 results.\n\n"
            "Red Sea disruptions added €45M in logistics costs over nine months. "
            "Resolution not expected before mid-2025.\n\n"
            "The stronger euro compressed Q3 revenue by approximately €120M. "
            "Forex headwinds are expected to persist into Q4."
        ),
        "source_type": "management",
        "source_date": date(2024, 11, 14),
    }
]

with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "ptc_pool.jsonl"
    
    extractor = PTCExtractor(backend=mock_backend)
    stats = extractor.extract_from_documents(documents, output_path=pool_path)
    
    ptcs = load_pool(pool_path)

print("Extraction stats:", stats)
print(f"\nPTCs in pool: {len(ptcs)}")
for p in ptcs:
    print(f"  [{p.direction.value:11s}] [{p.polarity:+d}]  {p.mechanism[:65]}...")

## Validation pipeline — errors are dropped, not raised

The extractor uses `validate_ptc_dict()` to convert raw LLM dicts to PTC objects. Invalid dicts (missing required field, bad polarity, event_date > source_date) are silently dropped and counted as errors. This is intentional: one bad PTC should not abort a 500-document run.

In [ ]:
from ptc_extraction.validation import validate_ptc_dict

# Valid
valid = validate_ptc_dict({
    "mechanism": "Rising raw material costs compress operating margins significantly.",
    "raw_text": "Raw material cost surge continues.",
    "direction": "precursor",
    "polarity": -1,
    "event_date": None,
    "confidence": 0.9,
}, source_id="test-src", source_date=date(2024, 1, 1),
   source_type="management", extracted_by="test", extraction_version="v1.1")
print("Valid PTC:", valid.mechanism[:50] if valid else None)

# Missing direction
missing_dir = validate_ptc_dict({
    "mechanism": "Something happened to the company.",
    "raw_text": "raw",
    "polarity": -1,
}, source_id="test-src", source_date=date(2024, 1, 1),
   source_type="management", extracted_by="test", extraction_version="v1.1")
print("Missing direction:", missing_dir)   # None — dropped

# event_date in the future
future_event = validate_ptc_dict({
    "mechanism": "Something will happen to the company performance next year.",
    "raw_text": "raw text here",
    "direction": "precursor",
    "polarity": 1,
    "event_date": "2025-12-31",   # AFTER source_date — invalid
    "confidence": 0.7,
}, source_id="test-src", source_date=date(2024, 1, 1),
   source_type="management", extracted_by="test", extraction_version="v1.1")
print("Future event_date:", future_event)   # None — dropped

## Resume-safe processing

If the extractor is interrupted mid-run, it resumes from where it left off. It tracks which `source_id`s are already in the pool and skips them on re-run. This is critical for a corpus of hundreds of documents.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "ptc_pool.jsonl"
    
    extractor = PTCExtractor(backend=mock_backend)
    
    # First run
    stats1 = extractor.extract_from_documents(documents, output_path=pool_path)
    count1 = mock_backend.complete.call_count
    
    # Second run — same document, resume=True (default)
    stats2 = extractor.extract_from_documents(documents, output_path=pool_path, resume=True)
    count2 = mock_backend.complete.call_count

print(f"LLM calls after first run  : {count1}")
print(f"LLM calls after second run : {count2}")
print(f"Extra calls on re-run      : {count2 - count1}  (should be 0 — source already in pool)")

## JSONL persistence and deduplication

In [ ]:
from ptc_extraction.io import load_pool, append_to_pool, dedup_pool

with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "pool.jsonl"
    
    extractor = PTCExtractor(backend=mock_backend)
    extractor.extract_from_documents(documents, output_path=pool_path)
    
    # Simulate a duplicate by appending the same batch again
    ptcs = load_pool(pool_path)
    append_to_pool(pool_path, ptcs)
    
    before = len(load_pool(pool_path))
    dedup_stats = dedup_pool(pool_path)
    after = len(load_pool(pool_path))

print(f"Before dedup : {before} PTCs")
print(f"After dedup  : {after} PTCs")
print(f"Removed      : {dedup_stats['removed']} duplicates")

---

## Key takeaways

1. The extractor chains chunker → LLM → validation → JSONL. Each layer is independently testable.
2. Validation drops bad LLM output silently — a bad PTC doesn't abort a long run.
3. Resume-safe processing means you can interrupt a 500-document corpus and restart without re-processing.
4. Dedup by `content_hash` catches identical mechanisms extracted from overlapping chunks.
5. The backend is swappable — mock in tests, Bedrock in production, same extractor code.

**Next:** Lesson 4 — Embedding and Clustering: building the canonical mechanism map.